# 06 · Classical Triangle-Hash Solver

Build the hash index from the brightest HYG stars, then solve a
synthetic image to verify the algorithm works at all before we
compare against the ML solver in notebook 07.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from src.data.catalog import load_hyg_catalog
from src.data.renderer import render_star_field
from src.models.classical_solver import ClassicalSolver, detect_centroids, triangle_invariant
catalog = load_hyg_catalog(ROOT / 'data/catalogs/hygdata_v3.csv', mag_limit=6.0)
print(f'Building index over {len(catalog)} bright stars...')

## Triangle invariant: the same triangle, rotated/scaled, hashes the same.

In [ ]:
tri = np.array([[0, 0], [3, 0], [1, 2]], dtype=float)
theta = np.deg2rad(37); rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
tri_rotscaled = (tri @ rot.T) * 2.7 + np.array([5, -3])
print('Original  invariant:', triangle_invariant(*tri))
print('Rot/scale invariant:', triangle_invariant(*tri_rotscaled))

In [ ]:
solver = ClassicalSolver(catalog, max_catalog_stars=300, max_triangle_arc_deg=20.0)
solver.build_index(verbose=True)

## Solve a synthetic image and visualize the matched centroids.

In [ ]:
img = render_star_field(85.0, -2.0, 25.0, 0.0, catalog,
                        image_size=224, rng=np.random.default_rng(0))
img_u8 = (img * 255).astype(np.uint8)
result = solver.solve(img_u8)
print(result)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), facecolor='#0b0d17')
ax.imshow(img); ax.set_facecolor('#0b0d17'); ax.set_xticks([]); ax.set_yticks([])
if len(result.centroids):
    ax.scatter(result.centroids[:, 0], result.centroids[:, 1],
               facecolors='none', edgecolors='#7c5cff', s=80, linewidths=1.2)
ax.set_title(f'Classical solve: RA={result.ra:.1f}° Dec={result.dec:.1f}° '
             f'(votes={result.n_matches}, t={result.solve_time*1000:.0f}ms)', color='white')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/06_classical_solve.png', dpi=150)
plt.show()